<a href="https://colab.research.google.com/github/Nghia-Trinh/Long-Short-Equity-Spring-26/blob/main/granite_4_0_tiny_base_preview.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 92.7 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


## Local Inference on GPU
Model page: https://huggingface.co/ibm-granite/granite-4.0-tiny-base-preview

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/ibm-granite/granite-4.0-tiny-base-preview)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-generation", model="ibm-granite/granite-4.0-tiny-base-preview")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("ibm-granite/granite-4.0-tiny-base-preview")
model = AutoModelForCausalLM.from_pretrained("ibm-granite/granite-4.0-tiny-base-preview")

In [ ]:
#FinnHub API
API_KEY = "d799t19r01qqpmhfvub0d799t19r01qqpmhfvubg"
API_SECRET = "d799t19r01qqpmhfvucg"

In [ ]:
import requests
import numpy as np
import pandas as pd

def get_stock_data(symbol):
    """Fetch basic price and news data from FinnHub."""
    url = f'https://finnhub.io/api/v1/quote?symbol={symbol}&token={API_KEY}'
    r = requests.get(url)
    return r.json()

def get_company_news(symbol):
    """Fetch recent news for risk sentiment analysis."""
    import datetime
    to_date = datetime.datetime.now().strftime('%Y-%m-%d')
    from_date = (datetime.datetime.now() - datetime.timedelta(days=7)).strftime('%Y-%m-%d')
    url = f'https://finnhub.io/api/v1/company-news?symbol={symbol}&from={from_date}&to={to_date}&token={API_KEY}'
    return requests.get(url).json()[:5] # Top 5 headlines

In [ ]:
def get_market_data(symbol):
    """Fetch real-time quote data (price, change, etc.) from FinnHub."""
    url = f'https://finnhub.io/api/v1/quote?symbol={symbol}&token={API_KEY}'
    response = requests.get(url)
    data = response.json()
    return {
        'symbol': symbol,
        'current_price': data.get('c'),
        'high': data.get('h'),
        'low': data.get('l'),
        'open': data.get('o'),
        'prev_close': data.get('pc')
    }

# Example usage:
# quote = get_market_data('AAPL')
# print(quote)

In [ ]:
def calculate_volatility_proxy(symbol):
    """Calculate price range as a volatility proxy using FinnHub quote data."""
    data = get_market_data(symbol)
    if not data['current_price'] or not data['high'] or not data['low']:
        return 0.0

    # Calculate the percentage range (High - Low) / Current Price
    price_range = (data['high'] - data['low']) / data['current_price']
    return round(price_range, 4)

# Example usage:
# vol = calculate_volatility_proxy('TSLA')
# print(f'Volatility Proxy: {vol * 100}%')

In [ ]:
def calculate_risk_adjustment(symbol, human_weight, news_summary):
    """Use Granite LLM to determine a risk multiplier based on sentiment."""
    prompt = f"""Analyze the following news for {symbol} and provide a risk adjustment multiplier between 0.9 and 1.1.
    A 1.0 means no change. 0.9 means high risk (reduce weight), 1.1 means low risk (increase weight).
    News: {news_summary}
    Return ONLY the number."""

    # Using the 'pipe' defined in previous cells
    res = pipe(prompt, max_new_tokens=10)
    try:
        # Simple parsing logic for the LLM output
        multiplier = float(res[0]['generated_text'].strip())
    except:
        multiplier = 1.0 # Default to no change if LLM output is messy

    adjusted_weight = human_weight * multiplier
    return round(adjusted_weight, 4)

In [ ]:
# Example: Stock Basket with Human-Generated Alpha Weights
portfolio = [
    {'symbol': 'AAPL', 'human_weight': 0.03}, # 3% target
    {'symbol': 'TSLA', 'human_weight': 0.05}, # 5% target
    {'symbol': 'NVDA', 'human_weight': 0.02}  # 2% target
]

print("--- Portfolio Risk Matrix Readjustment ---")
for stock in portfolio:
    news = get_company_news(stock['symbol'])
    headlines = " ".join([n.get('headline', '') for n in news])

    final_weight = calculate_risk_adjustment(stock['symbol'], stock['human_weight'], headlines)
    print(f"Stock: {stock['symbol']} | Human Weight: {stock['human_weight']*100}% | LLM Adjusted: {final_weight*100}%")